# 05 — Distributed training on CPU: DDP and FSDP

When a model does not fit in one machine's memory, or one machine is too slow, training is spread across several
processes, usually one per GPU, often on several machines. This lab shows **how to set that up** on a SageMaker
training job, with the two most common approaches: what changes in the job definition, and what changes in the
training script. It uses CPU instances so it runs in any account; the same code runs on GPU instances (with the
`nccl` backend instead of `gloo`).

| Strategy | What each process (rank) holds | What travels between ranks every step |
|---|---|---|
| `single` | the whole model, gradients and optimizer state; the whole batch | nothing (the baseline) |
| **DDP** (Distributed Data Parallel) | a full copy of everything; *its share* of the batch | all gradients, averaged across ranks (all-reduce) |
| **FSDP** (Fully Sharded Data Parallel) | *a shard* of the weights, gradients and optimizer state; its share of the batch | each layer's weights, gathered just before use and freed after (all-gather); gradients summed into the owning shard (reduce-scatter) |

DDP splits the **work**; FSDP also splits the **memory**, at the cost of more communication. That is why large
models are trained with FSDP (or DeepSpeed ZeRO, the same idea): it is what the multi-GPU recipes in
`workshops/distributed-training-deployment-on-sagemaker-ai/` use.

**What runs.** Three training jobs in parallel, each running `scripts/train_distributed.py` on the booking task
from notebook 01: `single` on 1 instance, `ddp` and `fsdp` on **2 instances** each. All three train the same
model with the same global batch (16 examples per step, 15 steps), so their losses should match closely; only
the way the work and memory are divided differs. This is a **full fine-tune** of Qwen3-0.6B, not LoRA: a LoRA
adapter is a few MB, so there would be nothing worth sharding. Each job reports seconds per step, peak memory and
parameters held per rank, and the fine-tuned model's exact-match score.

**How SageMaker runs it.** `instance_count=2` and `distributed=Torchrun()`: SageMaker starts both instances,
tells each one its rank and the address of the first, and launches the script with `torchrun`, one process per
instance (on GPU instances, one per GPU). The script joins the group with PyTorch's `gloo` backend, the CPU
counterpart of NCCL. Expected: ~8 minutes (measured: `single` and `ddp` ~4-5 min, `fsdp` ~7.5 min including
gathering the weights for evaluation).

**Quota:** 5 instances of a fallback type at the same time (`ml.c7i.4xlarge` and the others in `sm_helpers.py`).

In [ ]:
%pip install -q "sagemaker==3.21.0"

## Setup

As in notebooks 02-04: `Lab` (in `sm_helpers.py`) holds the session, role, bucket and MLflow; `CAPACITY` picks how
each job gets its instances (`"fallback"` by default; all instances of a job are of the same type).

In [ ]:
import sys

sys.path.insert(0, "scripts")
from sagemaker.train.distributed import Torchrun
from sm_helpers import Lab

# MLflow: "" = reuse the account's MLflow App (or create one: several minutes, first time only);
# "arn:aws:sagemaker:..." = use that app; None = off. Jobs run the same either way.
MLFLOW_APP_ARN = ""
lab = Lab(mlflow_app_arn=MLFLOW_APP_ARN)
CAPACITY = "fallback"   # "on-demand" | "spot" | "fallback"

## Train: one strategy per job

The script (`scripts/train_distributed.py`) is the same for all three; the `strategy` hyperparameter decides how
the model is wrapped:

* `ddp`: `DistributedDataParallel(model)`. Gradients are all-reduced during `backward()`
* `fsdp`: `fully_shard(layer)` for every decoder layer, then `fully_shard(model)` (PyTorch's FSDP2 API). The
  optimizer then only sees each rank's shards

Each rank takes its slice of the global batch (8 of the 16 examples with 2 ranks). After training, rank 0
gathers the full weights (FSDP) and scores the model on the 30 test requests.

In [ ]:
HP = {"steps": 15, "global_batch_size": 16, "learning_rate": 2e-5, "model_id": "Qwen/Qwen3-0.6B"}
jobs = {
    "single": lab.submit("dist-single", "train_distributed.py", {**HP, "strategy": "single"}, capacity=CAPACITY),
    "ddp": lab.submit("dist-ddp", "train_distributed.py", {**HP, "strategy": "ddp"}, capacity=CAPACITY,
                      instance_count=2, distributed=Torchrun()),
    "fsdp": lab.submit("dist-fsdp", "train_distributed.py", {**HP, "strategy": "fsdp"}, capacity=CAPACITY,
                       instance_count=2, distributed=Torchrun()),
}

In [ ]:
jobs = globals().get("jobs") or {k: lab.latest(f"dist-{k}") for k in ("single", "ddp", "fsdp")}   # after a kernel restart
desc = lab.wait(*jobs.values())

## Results

In [ ]:
print(f"{'strategy':9} {'ranks':>5} {'batch/rank':>10} {'sec/step':>8} {'peak memory per rank':>22} "
      f"{'params held per rank':>21} {'exact JSON':>10}")
results = {}
for name, job in jobs.items():
    if desc[job]["TrainingJobStatus"] != "Completed":
        print(f"{name:9} {desc[job]['TrainingJobStatus']}")
        continue
    m = results[name] = lab.metrics(job, "training_metrics.json")
    mem = " / ".join(f"{r['peak_rss_gb']:.1f}" for r in m["ranks"]) + " GB"
    held = " / ".join(f"{r['parameters_held_m']:.0f}" for r in m["ranks"]) + " M"
    print(f"{name:9} {m['world_size']:5} {m['per_rank_batch_size']:10} {m['sec_per_step']:8.1f} {mem:>22} "
          f"{held:>21} {m['finetuned']['exact_match']:10.0%}")
print("\nloss per step:")
for name, m in results.items():
    print(f"{name:9} " + " ".join(f"{v:.2f}" for v in m["loss"]))

### How to read the results

| Column | Meaning |
|---|---|
| ranks | processes in the job: one per instance here |
| batch/rank | examples each rank processes per step: the global batch (16) divided by the ranks |
| sec/step | wall time per training step, communication included (for reference) |
| peak memory per rank | the highest memory each process used, one value per rank |
| params held per rank | parameters stored on each rank, in millions: all ~600 M with `single` and DDP, about half each with FSDP |
| exact JSON | the fine-tuned model on the 30 test requests (same measure as notebook 01) |

**What to look for.**

* **Loss:** the three rows match (to two decimals in our runs). DDP and FSDP average gradients over the same 16
  examples that `single` sees in one process, so they compute the same update: distributing the work does not
  change what is learned. All three reach the same exact-match score.
* **Parameters held:** `single` and DDP keep all ~596 M parameters on every rank, along with their gradients and
  the optimizer's two moment estimates (~4x the weights for a full fine-tune with AdamW). FSDP keeps half on each
  of the 2 ranks; with 8 ranks it would be one eighth.
* **Memory** (measured: `single` ~21 GB, DDP ~17 GB per rank, FSDP ~12 GB per rank): DDP is lower than `single`
  only because each rank processes half the batch (fewer activations), not because it holds less. FSDP falls
  further, though by less than its parameter count, since activations, the layer being gathered and the process
  itself (PyTorch, tokenizer) are not sharded. This is how models that do not fit on one GPU get trained.
* **Speed** is not the point of this lab: every step sends gradients or weights between instances over the
  network, which on CPU instances can cost as much as the compute it saves. On GPU instances (NVLink, EFA) the
  same code is what makes large training runs faster.

## Track in MLflow

Each job's rank 0 logs the loss per step, seconds per step, rank 0's peak memory and the exact-match score to the
experiment `cpu-fine-tuning-techniques`, tagged `technique = distributed`.

In [ ]:
print(lab.mlflow_url() or "MLflow is off")

## Takeaway: the recipe

To make a training job distributed:

1. **Job:** set `instance_count` (here 2) and `distributed=Torchrun()` on the `ModelTrainer` (`Lab.submit` passes
   both). SageMaker starts the instances and launches the script with `torchrun` on each, one process per GPU (one
   per instance on CPU).
2. **Script:** join the process group (`dist.init_process_group("gloo")` on CPU, `"nccl"` on GPU), wrap the model
   (`DistributedDataParallel(model)` or `fully_shard(...)`), give each rank its share of every batch, and let only
   rank 0 write outputs. See `scripts/train_distributed.py`.
3. **Choose the strategy:**

   * **DDP** when the model, its gradients and optimizer state fit on each device: the simplest; it scales the
     batch, not the model.
   * **FSDP** when they don't: per-device memory falls with the number of ranks, at the price of more
     communication per step. It is what makes multi-billion-parameter fine-tuning fit.

The distributed-training workshop's recipes follow the same recipe with FSDP across the 4 GPUs of an
`ml.g5.12xlarge` (`Torchrun()` with one process per GPU).